In [53]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

current_path = Path.cwd()

PROJECT_ROOT = (
    current_path.parent
    if current_path.name == "notebooks"
    else current_path
)

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

bookings = pd.read_csv(
    PROCESSED_DATA_PATH / "hotel_bookings_clean.csv"
)

bookings.shape

(119210, 33)

In [54]:
arrival_date_text = (
    bookings["arrival_date_year"].astype(str)
    + " "
    + bookings["arrival_date_month"]
    + " "
    + bookings["arrival_date_day_of_month"].astype(str)
)

bookings["arrival_date"] = pd.to_datetime(
    arrival_date_text,
    format="%Y %B %d",
    errors="coerce"
)

In [55]:
bookings[
    [
        "arrival_date_year",
        "arrival_date_month",
        "arrival_date_day_of_month",
        "arrival_date"
    ]
].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date
0,2015,July,1,2015-07-01
1,2015,July,1,2015-07-01
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01


In [56]:
bookings["arrival_date"].isna().sum()

np.int64(0)

In [57]:
bookings["booking_date"] = (
    bookings["arrival_date"]
    - pd.to_timedelta(bookings["lead_time"], unit="D")
)

In [58]:
bookings[
    [
        "booking_date",
        "arrival_date",
        "lead_time"
    ]
].head()

,booking_date,arrival_date,lead_time
0,2014-07-24,2015-07-01,342
1,2013-06-24,2015-07-01,737
2,2015-06-24,2015-07-01,7
3,2015-06-18,2015-07-01,13
4,2015-06-17,2015-07-01,14


In [59]:
(
    bookings["booking_date"]
    > bookings["arrival_date"]
).sum()

np.int64(0)

In [60]:
bookings["booking_date"].min(), bookings["booking_date"].max()

(Timestamp('2013-06-24 00:00:00'), Timestamp('2017-08-31 00:00:00'))

In [61]:
leakage_columns = [
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list"
]

bookings = bookings.drop(
    columns=leakage_columns
)

In [62]:
bookings.shape

(119210, 30)

In [63]:
[column for column in leakage_columns
 if column in bookings.columns]

[]

In [64]:
bookings["children_missing"] = (
    bookings["children"].isna()
).astype(int)

In [65]:
bookings["has_agent"] = (
    bookings["agent"].notna()
).astype(int)

bookings["has_company"] = (
    bookings["company"].notna()
).astype(int)

In [66]:
bookings = bookings.drop(
    columns=["agent", "company"]
)

In [67]:
bookings["total_stay_nights"] = (
    bookings["stays_in_weekend_nights"]
    + bookings["stays_in_week_nights"]
)

In [68]:
(bookings["total_stay_nights"] == 0).sum()

np.int64(645)

In [69]:
zero_stay_bookings = bookings[
    bookings["total_stay_nights"] == 0
].copy()

In [70]:
zero_stay_bookings["is_canceled"].value_counts()

is_canceled
0    622
1     23
Name: count, dtype: int64

In [71]:
(
    zero_stay_bookings["is_canceled"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

is_canceled
0    96.43
1     3.57
Name: proportion, dtype: float64

In [72]:
bookings["is_zero_stay_booking"] = (
    bookings["total_stay_nights"] == 0
).astype(int)

In [73]:
bookings["weekend_night_ratio"] = np.where(
    bookings["total_stay_nights"] > 0,
    bookings["stays_in_weekend_nights"]
    / bookings["total_stay_nights"],
    0
)

In [74]:
bookings["total_guests"] = (
    bookings["adults"]
    + bookings["children"].fillna(0)
    + bookings["babies"]
)

In [75]:
(bookings["total_guests"] == 0).sum()

np.int64(0)

In [76]:
bookings["is_family_booking"] = (
    (
        bookings["children"].fillna(0)
        + bookings["babies"]
    ) > 0
).astype(int)

In [77]:
bookings["weekend_night_ratio"] = (
    bookings["stays_in_weekend_nights"]
    / bookings["total_stay_nights"]
)

In [78]:
bookings["booking_month"] = (
    bookings["booking_date"].dt.month
)

bookings["booking_day_of_week"] = (
    bookings["booking_date"].dt.dayofweek
)

In [79]:
bookings["booking_season"] = np.select(
    [
        bookings["booking_month"].isin([12, 1, 2]),
        bookings["booking_month"].isin([3, 4, 5]),
        bookings["booking_month"].isin([6, 7, 8]),
        bookings["booking_month"].isin([9, 10, 11])
    ],
    [
        "Winter",
        "Spring",
        "Summer",
        "Autumn"
    ],
    default="Unknown"
)

In [80]:
bookings[
    [
        "booking_date",
        "booking_month",
        "booking_day_of_week",
        "booking_season"
    ]
].head()

,booking_date,booking_month,booking_day_of_week,booking_season
0,2014-07-24,7,3,Summer
1,2013-06-24,6,0,Summer
2,2015-06-24,6,2,Summer
3,2015-06-18,6,3,Summer
4,2015-06-17,6,2,Summer


In [81]:
bookings["estimated_booking_value"] = (
    bookings["adr"]
    * bookings["total_stay_nights"]
)

In [82]:
bookings[
    [
        "adr",
        "total_stay_nights",
        "estimated_booking_value"
    ]
].head()

,adr,total_stay_nights,estimated_booking_value
0,0.0,0,0.0
1,0.0,0,0.0
2,75.0,1,75.0
3,75.0,1,75.0
4,98.0,2,196.0


In [83]:
outlier_features = [
    "lead_time",
    "adr",
    "total_stay_nights",
    "total_guests",
    "estimated_booking_value",
    "previous_cancellations"
]

outlier_summary = pd.DataFrame({
    "minimum": bookings[outlier_features].min(),
    "p01": bookings[outlier_features].quantile(0.01),
    "median": bookings[outlier_features].median(),
    "p99": bookings[outlier_features].quantile(0.99),
    "maximum": bookings[outlier_features].max()
})

outlier_summary

,minimum,p01,median,p99,maximum
lead_time,0.0,0.0,69.00,444.0,737.0
adr,0.0,0.0,94.95,252.0,5400.0
total_stay_nights,0.0,1.0,3.00,14.0,69.0
total_guests,1.0,1.0,2.00,4.0,55.0
estimated_booking_value,0.0,0.0,267.00,1680.0,7590.0
previous_cancellations,0.0,0.0,0.00,1.0,26.0


In [84]:
bookings.sort_values(
    "estimated_booking_value",
    ascending=False
)[
    [
        "hotel",
        "lead_time",
        "total_stay_nights",
        "total_guests",
        "adr",
        "estimated_booking_value",
        "is_canceled"
    ]
].head(10)

,hotel,lead_time,total_stay_nights,total_guests,adr,estimated_booking_value,is_canceled
14031,Resort Hotel,126,69,1.0,110.00,7590.00,0
14030,Resort Hotel,113,60,1.0,110.50,6630.00,0
13135,Resort Hotel,378,14,2.0,450.00,6300.00,1
60101,City Hotel,16,29,2.0,212.00,6148.00,1
48491,City Hotel,35,1,2.0,5400.00,5400.00,1
60714,City Hotel,65,29,4.0,171.30,4967.70,1
64337,City Hotel,3,21,1.0,231.81,4868.01,1
27235,Resort Hotel,82,14,4.0,318.00,4452.00,0
62547,City Hotel,123,30,3.0,140.00,4200.00,1
60747,City Hotel,66,28,4.0,147.90,4141.20,1


In [85]:
bookings[
    [
        "arrival_date",
        "booking_date",
        "lead_time",
        "adr",
        "estimated_booking_value",
        "is_canceled"
    ]
].dtypes

arrival_date               datetime64[us]
booking_date               datetime64[us]
lead_time                           int64
adr                               float64
estimated_booking_value           float64
is_canceled                         int64
dtype: object

In [86]:
(bookings["adr"] < 0).sum()

np.int64(0)

In [87]:
numeric_data = bookings.select_dtypes(
    include=np.number
)

np.isinf(numeric_data).sum().sort_values(
    ascending=False
).head(10)

is_canceled                  0
lead_time                    0
arrival_date_year            0
arrival_date_week_number     0
arrival_date_day_of_month    0
stays_in_weekend_nights      0
stays_in_week_nights         0
adults                       0
children                     0
babies                       0
dtype: int64

In [88]:
excluded_from_model = [
    "is_canceled",
    "arrival_date",
    "booking_date"
]

In [89]:
forbidden_columns = leakage_columns + excluded_from_model

[column for column in forbidden_columns
 if column not in bookings.columns]

['reservation_status',
 'reservation_status_date',
 'assigned_room_type',
 'booking_changes',
 'days_in_waiting_list']

In [90]:
bookings.columns.tolist()

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'deposit_type',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'adr_was_negative',
 'arrival_date',
 'booking_date',
 'children_missing',
 'has_agent',
 'has_company',
 'total_stay_nights',
 'is_zero_stay_booking',
 'weekend_night_ratio',
 'total_guests',
 'is_family_booking',
 'booking_month',
 'booking_day_of_week',
 'booking_season',
 'estimated_booking_value']

In [91]:
bookings.shape

(119210, 40)

In [92]:
bookings.isna().sum().sort_values(
    ascending=False )

weekend_night_ratio               645
country                           478
children                            4
adr                                 1
estimated_booking_value             1
hotel                               0
lead_time                           0
is_canceled                         0
stays_in_weekend_nights             0
arrival_date_day_of_month           0
arrival_date_week_number            0
arrival_date_month                  0
arrival_date_year                   0
meal                                0
babies                              0
adults                              0
distribution_channel                0
is_repeated_guest                   0
previous_bookings_not_canceled      0
previous_cancellations              0
reserved_room_type                  0
deposit_type                        0
market_segment                      0
stays_in_week_nights                0
required_car_parking_spaces         0
customer_type                       0
total_of_spe

In [93]:
bookings.to_csv(
    PROCESSED_DATA_PATH / "hotel_booking_features.csv",
    index=False
)